In [9]:
import tensorflow as tf
from tensorflow.keras import layers, Model

# Load MNIST
(x_train, _), (_, _) = tf.keras.datasets.mnist.load_data()
x_train = (x_train.astype("float32") / 255.0)
x_train = x_train[..., None]

# Noise schedule
T = 100
beta = tf.linspace(0.0001, 0.02, T)
alpha = 1.0 - beta
alpha_bar = tf.math.cumprod(alpha)

# Model inputs
image_input = layers.Input(shape=(28, 28, 1))
time_input = layers.Input(shape=(1,))

# Convert timestep into a feature map
t = layers.Dense(28 * 28, activation="relu")(time_input)
t = layers.Reshape((28, 28, 1))(t)

# Combine noisy image with timestep information
x = layers.Concatenate()([image_input, t])

# CNN predicts noise
x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
noise_output = layers.Conv2D(1, 3, padding="same")(x)

model = Model([image_input, time_input], noise_output)
model.compile(optimizer="adam", loss="mse")

In [ ]:
optimizer = tf.keras.optimizers.Adam()

@tf.function
def train_step(images):
    batch_size = tf.shape(images)[0]

    # Random timestep for each image
    t = tf.random.uniform(
        (batch_size,), minval=0, maxval=T, dtype=tf.int32
    )

    noise = tf.random.normal(tf.shape(images))

    a_bar = tf.gather(alpha_bar, t)
    a_bar = tf.reshape(a_bar, (-1, 1, 1, 1))

    # Add noise to the original images
    noisy_images = (
        tf.sqrt(a_bar) * images +
        tf.sqrt(1.0 - a_bar) * noise
    )

    time_input = tf.cast(t[:, None], tf.float32) / T

    with tf.GradientTape() as tape:
        predicted_noise = model(
            [noisy_images, time_input], training=True
        )
        loss = tf.reduce_mean(tf.square(noise - predicted_noise))

    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(
        zip(gradients, model.trainable_variables)
    )

    return loss

# Train
for epoch in range(5):
    losses = []

    for batch in tf.data.Dataset.from_tensor_slices(
        x_train
    ).shuffle(10000).batch(128):
        losses.append(train_step(batch))

    print(f"Epoch {epoch+1}: loss = {tf.reduce_mean(losses):.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Start with random noise
generated = tf.random.normal((1, 28, 28, 1))

for t in reversed(range(T)):
    time_input = tf.constant([[t / T]], dtype=tf.float32)

    predicted_noise = model(
        [generated, time_input], training=False
    )

    a = alpha[t]
    a_bar = alpha_bar[t]
    b = beta[t]

    generated = (
        (generated - (b / tf.sqrt(1.0 - a_bar)) * predicted_noise)
        / tf.sqrt(a)
    )

    # Add noise except at the final step
    if t > 0:
        generated += tf.sqrt(b) * tf.random.normal(tf.shape(generated))

# Display generated image
image = (generated[0, :, :, 0] + 1) / 2

plt.imshow(image, cmap="gray")
plt.axis("off")
plt.show()